# 04 — Disengagement Feature Engineering
SpiriCom · Huawei Technologies Tunisia · PFE 2026

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 60)
PROC_DIR  = Path('data/processed')
MODEL_DIR = Path('models')
OUT_DIR   = Path('data/outputs')

churn = pd.read_parquet(PROC_DIR / 'churn_labelled_v6.parquet')
blocklist = json.load(open(MODEL_DIR / 'leakage_blocklist.json'))
BLOCKED = set(blocklist['blocked_features'])
print(f'Loaded: {churn.shape[0]:,} rows x {churn.shape[1]} cols')
print(f'Blocklist: {len(BLOCKED)} features')

# NB04-1: labelled population only
data = churn.dropna(subset=['churn']).copy()
data['churn'] = data['churn'].astype(int)
print(f'Labelled rows: {len(data):,} '
      f'(disengaged={int(data["churn"].sum()):,}, '
      f'rate={data["churn"].mean()*100:.1f}%)')


Loaded: 4,896 rows x 34 cols
Blocklist: 31 features
Labelled rows: 2,566 (disengaged=868, rate=33.8%)


## 1 — De-imputation of QoS columns

In [2]:
# Objective:
# Reverse artificial imputation applied during preprocessing (NB03),
# and restore missingness as a meaningful signal.
#
# Key idea:
# Instead of treating imputed values as real measurements,
# we explicitly convert them back to NaN to preserve data integrity.
#
# Rationale:
# Imputation introduces bias in:
#   → distribution statistics
#   → churn threshold computation
#   → behavioral interpretation
#
# Therefore:
#   → imputed values are removed
#   → missingness becomes an informative feature
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Identify imputation indicator columns
# ────────────────────────────────────────────────────────────────
# These flags were created in NB03-2 spike detection phase

imp_flag_cols = [
    c for c in data.columns
    if c.endswith('_imputed')
]


# ────────────────────────────────────────────────────────────────
# STEP 2 — Restore missing values (de-imputation process)
# ────────────────────────────────────────────────────────────────
# For each imputed column:
#   - locate rows marked as imputed
#   - replace synthetic value with NaN
#   - preserve missingness structure for modeling

restored = []

for flag in imp_flag_cols:

    base = flag[:-len('_imputed')]

    # only process valid feature columns
    if base in data.columns and base not in BLOCKED:

        n = int((data[flag] == 1).sum())

        # reverse imputation → restore missing values
        data.loc[data[flag] == 1, base] = np.nan

        restored.append((base, n))


# ────────────────────────────────────────────────────────────────
# STEP 3 — Audit restored missingness
# ────────────────────────────────────────────────────────────────
# This step quantifies the level of artificial data correction removal

print('Median-imputed values removed (restored to NaN):')

for base, n in restored:
    print(
        f'  {base:<28s}: {n:>5,} rows → NaN '
        f'({data[base].isna().mean()*100:.1f}% missing now)'
    )


# ────────────────────────────────────────────────────────────────
# STEP 4 — Preserve missingness signals for ML models
# ────────────────────────────────────────────────────────────────
# Important modeling insight:
# Missingness is NOT noise — it is often:
#   → network failure indicator
#   → logging inconsistency
#   → low-quality session proxy

print('\nImputation flags retained as predictive features:')
print([f for f in imp_flag_cols])

Median-imputed values removed (restored to NaN):
  e2e_delay_ms                :   315 rows → NaN (12.3% missing now)
  client_rtt_ms               :   321 rows → NaN (12.5% missing now)

Imputation flags retained as predictive features:
['dou_total_imputed', 'duration_imputed', 'traffic_5g_imputed', 'traffic_4g_imputed', 'traffic_3g_imputed', 'traffic_2g_imputed', 'e2e_delay_ms_imputed', 'client_rtt_ms_imputed']


## 2 — Device & capability features 

In [3]:
# Objective:
# Encode device generation into binary capability features
# to capture technology maturity and its impact on disengagement.
#
# Rationale:
# Network generation is a proxy for:
#   → QoS potential (latency, throughput)
#   → service accessibility (VoLTE / NR availability)
#   → user experience quality
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Normalize generation field
# ────────────────────────────────────────────────────────────────
gen = (
    data['generation']
    .fillna('UNKNOWN')
    .astype(str)
    .str.upper()
)


# ────────────────────────────────────────────────────────────────
# STEP 2 — Binary capability encoding
# ────────────────────────────────────────────────────────────────
# Each feature encodes presence of a network technology class.

data['is_2g_only'] = (gen == '2G').astype(int)

data['has_3g'] = gen.str.contains('3G').astype(int)

data['has_lte'] = gen.str.contains('LTE').astype(int)

data['has_nr'] = gen.str.contains('NR').astype(int)


# ────────────────────────────────────────────────────────────────
# STEP 3 — Behavioral validation (disengagement rate)
# ────────────────────────────────────────────────────────────────
# We compute conditional churn probability:
#
#   P(churn | feature = 1)
#
# This measures how network capability correlates with disengagement.

print('Capability flags vs disengagement rate:')

for f in ['is_2g_only', 'has_3g', 'has_lte', 'has_nr']:

    g = data.groupby(f)['churn'].agg(['mean', 'count'])

    print(f'\n{f}:')

    print(
        (g['mean'] * 100).round(1).astype(str) + '%  (n=' +
        g['count'].astype(str) + ')'
    )


# ────────────────────────────────────────────────────────────────
# STEP 4 — Multicollinearity detection (critical for ML models)
# ────────────────────────────────────────────────────────────────
# We detect deterministic redundancy:
#
#   has_3g ≈ 1 - is_2g_only
#
# This creates perfect linear dependence → problematic for:
#   - Logistic Regression coefficients
#   - SHAP interpretation stability

_is_complement = (data['has_3g'] == 1 - data['is_2g_only']).all()

if _is_complement:
    print(
        '\nNB04-FIX-2: has_3g == 1 - is_2g_only (perfect complement) '
        '→ has_3g excluded from feature matrix to avoid multicollinearity'
    )


# ────────────────────────────────────────────────────────────────
# STEP 5 — Brand encoding (categorical compression)
# ────────────────────────────────────────────────────────────────
# Strategy:
#   - Keep top 10 most frequent brands
#   - Collapse others into "OTHER"
#   - Apply one-hot encoding

top_brands = data['brand'].value_counts().head(10).index

data['brand_grp'] = np.where(
    data['brand'].isin(top_brands),
    data['brand'],
    'OTHER'
)

brand_oh = pd.get_dummies(
    data['brand_grp'],
    prefix='brand',
    dtype=int
)

print(f'\nBrand one-hot features: {list(brand_oh.columns)}')

Capability flags vs disengagement rate:

is_2g_only:
is_2g_only
0    30.2%  (n=2419)
1     93.2%  (n=147)
dtype: object

has_3g:
has_3g
0     93.2%  (n=147)
1    30.2%  (n=2419)
dtype: object

has_lte:
has_lte
0     85.6%  (n=174)
1    30.1%  (n=2392)
dtype: object

has_nr:
has_nr
0    34.9%  (n=2146)
1     28.3%  (n=420)
dtype: object

NB04-FIX-2: has_3g == 1 - is_2g_only (perfect complement) → has_3g excluded from feature matrix to avoid multicollinearity

Brand one-hot features: ['brand_APPLE', 'brand_HONOR', 'brand_HUAWEI', 'brand_INFINIX', 'brand_ITEL', 'brand_OPPO', 'brand_OTHER', 'brand_REALME', 'brand_SAMSUNG', 'brand_TECNO', 'brand_XIAOMI']


## 3 — Geography & profile features

In [4]:
# Objective:
# Encode spatial + demographic profile information into ML-ready signals
# capturing behavioral heterogeneity across regions and user segments.
#
# Rationale:
# Geography and user profile act as latent proxies for:
#   → infrastructure quality differences
#   → socio-economic segmentation
#   → mobility behavior patterns
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Initialize feature container
# ────────────────────────────────────────────────────────────────
# We progressively build a list of one-hot encoded feature blocks

oh_frames = [brand_oh]


# ────────────────────────────────────────────────────────────────
# STEP 2 — Geographic segmentation (Zone encoding)
# ────────────────────────────────────────────────────────────────
# We reduce high-cardinality zones (24 regions) into:
#   - Top 12 most frequent zones
#   - OTHER category for long tail

if 'layer2name' in data.columns:

    top_zones = data['layer2name'].value_counts().head(12).index

    data['zone_grp'] = np.where(
        data['layer2name'].isin(top_zones),
        data['layer2name'],
        'OTHER'
    )

    oh_frames.append(
        pd.get_dummies(data['zone_grp'], prefix='zone', dtype=int)
    )


# ────────────────────────────────────────────────────────────────
# STEP 3 — Low-cardinality profile features
# ────────────────────────────────────────────────────────────────
# These variables represent user behavioral segmentation:
#
#   mobility_class → movement intensity
#   usertype       → subscription category
#   user_class      → segmentation tier
#   tertype         → service type classification

for col in ['mobility_class', 'usertype', 'user_class', 'tertype']:

    if col in data.columns:

        vals = (
            data[col]
            .fillna('UNKNOWN')
            .astype(str)
            .str.upper()
        )

        # Only encode if variable is low-cardinality
        if vals.nunique() <= 10:

            oh_frames.append(
                pd.get_dummies(vals, prefix=col, dtype=int)
            )

            print(f'{col}: {vals.nunique()} categories one-hot encoded')


# ────────────────────────────────────────────────────────────────
# STEP 4 — Feature matrix assembly
# ────────────────────────────────────────────────────────────────
# Combine all categorical encodings into a single matrix

oh = pd.concat(oh_frames, axis=1).reset_index(drop=True)


# ────────────────────────────────────────────────────────────────
# STEP 5 — Data quality control (constant features removal)
# ────────────────────────────────────────────────────────────────
# Constant columns provide no information gain and must be removed

const = [c for c in oh.columns if oh[c].nunique() <= 1]

if const:
    print(f'Dropped constant one-hot columns: {const}')
    oh = oh.drop(columns=const)


# ────────────────────────────────────────────────────────────────
# STEP 6 — Final diagnostic
# ────────────────────────────────────────────────────────────────
print(f'\nTotal one-hot encoded features: {oh.shape[1]}')

mobility_class: 3 categories one-hot encoded
usertype: 1 categories one-hot encoded
user_class: 7 categories one-hot encoded
tertype: 2 categories one-hot encoded
Dropped constant one-hot columns: ['usertype_DATA USER']

Total one-hot encoded features: 36


## 4 — Assemble the feature matrix (blocklist enforced)

In [5]:
# Objective:
# Construct the final ML-ready feature matrix X while enforcing:
#   → leakage prevention
#   → multicollinearity control
#   → zero-variance elimination
#
# Output:
#   X = feature matrix (numeric + categorical + flags)
#   y = churn label (v6 disengagement definition)
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Select valid numeric QoS features
# ────────────────────────────────────────────────────────────────
# These represent network quality and service experience KPIs.
# All features must be:
#   - present in dataset
#   - not explicitly blocked due to leakage risk

NUMERIC_CANDIDATES = [
    c for c in [
        'e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms',
        'client_packet_loss_rate', 'server_packet_loss_rate',
        'page_response_delay', 'page_download_throughput',
        'dns_delay', 'dns_sr', 'tcp_connection_sr',
        'voip_voice_downlink_throughput',
        'voip_voice_uplink_throughput',
        'number_of_regions',
    ]
    if c in data.columns and c not in BLOCKED
]


# ────────────────────────────────────────────────────────────────
# STEP 2 — Imputation flag handling (controlled removal)
# ────────────────────────────────────────────────────────────────
# Rationale:
#   - dou_total_imputed / duration_imputed are structural artifacts
#   - they are constant-zero in labelled population
#   - therefore they introduce zero variance and no predictive value

LABEL_IMP_FLAGS = {
    'dou_total_imputed',
    'duration_imputed'
}


# ────────────────────────────────────────────────────────────────
# STEP 3 — Capability flags (reduced collinearity design)
# ────────────────────────────────────────────────────────────────
# NB04-FIX-2:
#   has_3g = 1 - is_2g_only (perfect multicollinearity)
#   → removed from feature space

FLAG_FEATURES = (
    [
        c for c in data.columns
        if c.endswith('_imputed') and c not in LABEL_IMP_FLAGS
    ]
    + ['is_2g_only', 'has_lte', 'has_nr']
)


# ────────────────────────────────────────────────────────────────
# STEP 4 — Feature matrix construction
# ────────────────────────────────────────────────────────────────
# Concatenate:
#   1. Numeric QoS features
#   2. Binary capability + imputation flags
#   3. One-hot encoded categorical features

X = pd.concat(
    [
        data[NUMERIC_CANDIDATES + FLAG_FEATURES].reset_index(drop=True),
        oh.reset_index(drop=True)
    ],
    axis=1
)

y = data['churn'].reset_index(drop=True)
msisdn = data['msisdn'].reset_index(drop=True)


# ────────────────────────────────────────────────────────────────
# STEP 5 — Leakage enforcement (hard constraint)
# ────────────────────────────────────────────────────────────────
# Mathematical constraint:
#   No feature in X may belong to BLOCKED set:
#
#   BLOCKED ⟂ X   (strict independence requirement)

leaks = [c for c in X.columns if c in BLOCKED]
assert not leaks, f'BLOCKED FEATURES IN X: {leaks}'


# ────────────────────────────────────────────────────────────────
# STEP 6 — Zero-variance elimination (robustness layer)
# ────────────────────────────────────────────────────────────────
# Remove features with:
#   Var(X_j) = 0 → no information gain

const_X = [c for c in X.columns if X[c].nunique() <= 1]

if const_X:
    print(f'NB04-FIX-3: dropped constant columns: {const_X}')
    X = X.drop(columns=const_X)
else:
    print('NB04-FIX-3: zero-variance check passed — no constant columns.')


# ────────────────────────────────────────────────────────────────
# STEP 7 — Final dataset summary
# ────────────────────────────────────────────────────────────────
print(f'Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features')
print(f'  numeric QoS   : {len(NUMERIC_CANDIDATES)}')
print(f'  flags         : {len(FLAG_FEATURES)}')
print(f'  one-hot       : {oh.shape[1]}')
print(f'  missing values: {int(X.isna().sum().sum()):,} '
      '(intentionally preserved for tree-based models)')

NB04-FIX-3: zero-variance check passed — no constant columns.
Feature matrix: 2,566 rows × 56 features
  numeric QoS   : 11
  flags         : 9
  one-hot       : 36
  missing values: 636 (intentionally preserved for tree-based models)


## 5 — Signal preview
Univariate check before modeling: point-biserial correlation for numeric features,
disengagement rate per one-hot group. Expect **modest** values — the strong volume
signals are blocked by design; what remains is the honest experience/device/geo signal.

In [6]:
# Objective:
# Perform pre-modeling sanity checks to evaluate:
#   → linear signal strength (numeric features)
#   → segmentation power (categorical features)
#
# Important note:
# Strong leakage-driven variables were intentionally removed in NB03–NB04.
# Therefore, remaining correlations reflect *true behavioral signal*.
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Point-biserial correlation (numeric vs churn)
# ────────────────────────────────────────────────────────────────
# Interpretation:
#   r ∈ [-1, 1]
#   measures linear association between continuous KPI and binary churn

print('Numeric features — point-biserial correlation with disengagement:')

rows = []

for c in NUMERIC_CANDIDATES:

    valid = X[c].notna()

    # ensure statistical validity
    if valid.sum() > 100 and X.loc[valid, c].std() > 0:

        r = np.corrcoef(
            X.loc[valid, c],
            y[valid]
        )[0, 1]

        rows.append((c, r, int(valid.sum())))


# ────────────────────────────────────────────────────────────────
# STEP 2 — Ranking by absolute correlation strength
# ────────────────────────────────────────────────────────────────
# We highlight strongest linear signals (positive or negative)

for c, r, n in sorted(rows, key=lambda t: -abs(t[1])):

    bar = '#' * int(abs(r) * 50)

    print(
        f'  {c:<34s} r={r:>7.3f}  n={n:>5,}  {bar}'
    )


# ────────────────────────────────────────────────────────────────
# STEP 3 — One-hot signal diagnostic (categorical impact)
# ────────────────────────────────────────────────────────────────
# We compute:
#
#   P(churn | category = 1)
#
# and compare it to global churn rate:
#
#   Δ = P(churn | category) − P(churn)

global_rate = y.mean() * 100

print(
    '\nTop one-hot groups by disengagement-rate gap '
    f'vs average ({global_rate:.1f}%):'
)

gaps = []

for c in oh.columns:

    m = (oh[c] == 1).values  # NB04-fix: positional mask (index-safe)

    if m.sum() >= 30:

        gaps.append((
            c,
            y[m].mean() * 100,
            int(m.sum())
        ))


# ────────────────────────────────────────────────────────────────
# STEP 4 — Ranking categorical signals
# ────────────────────────────────────────────────────────────────
# We select categories with strongest deviation from global behavior

for c, rate, n in sorted(gaps, key=lambda t: -abs(t[1] - global_rate))[:12]:

    print(
        f'  {c:<34s} {rate:>5.1f}%  (n={n:,})'
    )

Numeric features — point-biserial correlation with disengagement:
  tcp_connection_sr                  r= -0.523  n=2,566  ##########################
  dns_sr                             r= -0.387  n=2,566  ###################
  number_of_regions                  r= -0.281  n=2,566  ##############
  client_packet_loss_rate            r= -0.113  n=2,566  #####
  client_rtt_ms                      r=  0.098  n=2,245  ####
  dns_delay                          r= -0.095  n=2,566  ####
  server_rtt_ms                      r=  0.075  n=2,566  ###
  page_download_throughput           r= -0.074  n=2,566  ###
  e2e_delay_ms                       r=  0.066  n=2,251  ###
  server_packet_loss_rate            r= -0.066  n=2,566  ###
  page_response_delay                r= -0.048  n=2,566  ##

Top one-hot groups by disengagement-rate gap vs average (33.8%):
  tertype_FEATUREPHONE                97.1%  (n=136)
  user_class_OTHER_TRAFFIC            91.3%  (n=495)
  brand_OTHER                         

## 6 — Persist features + reproducible split

In [7]:

# Objective:
# Persist the engineered dataset and create a reproducible
# train/test split for downstream modeling (NB05).
#
# Key principles:
#   → reproducibility (fixed seed)
#   → stratification (class balance preservation)
#   → traceability (metadata logging)
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Stratified train/test split
# ────────────────────────────────────────────────────────────────
# Stratification ensures:
#   P(churn) train ≈ P(churn) test

X_tr, X_te, y_tr, y_te, id_tr, id_te = train_test_split(
    X,
    y,
    msisdn,
    test_size=0.25,
    stratify=y,
    random_state=42
)


# ────────────────────────────────────────────────────────────────
# STEP 2 — Build persisted dataset
# ────────────────────────────────────────────────────────────────
# We attach:
#   - features (X)
#   - target (y)
#   - identifier (msisdn)
#   - split indicator

out = X.copy()
out['churn'] = y
out['msisdn'] = msisdn

out['split'] = np.where(
    out['msisdn'].isin(set(id_te)),
    'test',
    'train'
)


# ────────────────────────────────────────────────────────────────
# STEP 3 — Save feature dataset
# ────────────────────────────────────────────────────────────────
# Output is the single source of truth for NB05 modeling

feat_path = PROC_DIR / 'churn_features_v6.parquet'
out.to_parquet(feat_path, index=False)


# ────────────────────────────────────────────────────────────────
# STEP 4 — Metadata logging (reproducibility layer)
# ────────────────────────────────────────────────────────────────
# This ensures full experiment traceability:
#   → feature list
#   → split configuration
#   → label version
#   → dataset size

meta = {
    'generated_at': datetime.now().isoformat(),
    'label_version': 'v6',

    'n_rows': int(len(out)),
    'n_features': int(X.shape[1]),

    'feature_columns': list(X.columns),
    'numeric_qos': NUMERIC_CANDIDATES,
    'flag_features': FLAG_FEATURES,

    'split': {
        'train': int((out['split'] == 'train').sum()),
        'test': int((out['split'] == 'test').sum()),
        'stratified': True,
        'random_state': 42
    },

    'disengaged_rate': round(float(y.mean()), 4),
    'blocklist_enforced': True
}


# Save metadata JSON
json.dump(
    meta,
    open(MODEL_DIR / 'churn_features_meta.json', 'w'),
    indent=2
)


# ────────────────────────────────────────────────────────────────
# STEP 5 — Final audit
# ────────────────────────────────────────────────────────────────
print(f'Saved dataset : {feat_path} ({out.shape[0]:,} × {out.shape[1]})')
print(f'Metadata file : {MODEL_DIR / "churn_features_meta.json"}')

print(
    f'Train {meta["split"]["train"]:,} / '
    f'Test {meta["split"]["test"]:,} '
    f'(stratified, seed 42)'
)


# ────────────────────────────────────────────────────────────────
# STEP 6 — Modeling expectations (important for thesis framing)
# ────────────────────────────────────────────────────────────────
print('\nNext → 05_Churn_Modeling: Logistic Regression / Random Forest / XGBoost')

print('\nExpected behavior:')
print('- Volume-based leakage features removed → reduced linear separability')
print('- Performance expected to be moderate but stable')
print('- Evaluation should focus on PR-AUC and lift vs baseline')
print('- SHAP analysis will provide main business insights')

Saved dataset : data\processed\churn_features_v6.parquet (2,566 × 59)
Metadata file : models\churn_features_meta.json
Train 1,924 / Test 642 (stratified, seed 42)

Next → 05_Churn_Modeling: Logistic Regression / Random Forest / XGBoost

Expected behavior:
- Volume-based leakage features removed → reduced linear separability
- Performance expected to be moderate but stable
- Evaluation should focus on PR-AUC and lift vs baseline
- SHAP analysis will provide main business insights
